# Milestone 1: Maximizing Inference with Majority Voting

## 1. Configuration and Setup

This cell contains all the configuration variables. We define the model, data paths, and a new output path for our aggregated results.

In [2]:
import json
import os
import re
from collections import Counter
from typing import List, Optional

from tqdm import tqdm
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

# ─── Configuration ─────────────────────────────────────────────────
MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID = "0"
DATA_PATH = "data/public.jsonl"
OUTPUT_PATH = "results/milestone1_results.jsonl" # New output file for this strategy

# Set the device environment variable
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

## 2. Data Loading and Prompt Engineering

We load the dataset and define the prompt-building functions exactly as in the baseline. This part remains unchanged.

In [3]:
data = [json.loads(line) for line in open(DATA_PATH)]
print(f"Loaded {len(data)} questions.")

SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician and a meticulous problem-solver. "
    "Solve the problem thoroughly, showing all necessary steps and reasoning clearly. "
    "Crucially, present your FINAL answer inside a single \\\\boxed{} block. "
    "If the problem requires multiple numerical sub-answers, list them separated by commas inside that single \\\\boxed{} block, "
    "e.g., \\\\boxed{3, 7}. Ensure no other \\\\boxed{} appears in your response except for the final answer.\n"
    "\n"
    "Here's an example:\n"
    "Problem: What is 5 times 3 plus 2?\n"
    "Solution:\n"
    "Let the problem be expressed as an equation: $5 \\times 3 + 2$.\n"
    "First, perform the multiplication: $5 \\times 3 = 15$.\n"
    "Next, perform the addition: $15 + 2 = 17$.\n"
    "The final answer is 17.\n"
    "\\\\boxed{17}"
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Carefully read the problem and the answer choices below. "
    "First, reason step-by-step to determine the single best answer. "
    "Then, and ONLY then, output the letter of your chosen option inside a \\\\boxed{} block, "
    "e.g., \\\\boxed{C}. Do not include any other text or explanation after the \\\\boxed{} answer.\n"
    "\n"
    "Here's an example:\n"
    "Problem: Which of the following is an even number?\n"
    "Options:\n"
    "A. 3\n"
    "B. 7\n"
    "C. 4\n"
    "D. 9\n"
    "Solution:\n"
    "An even number is an integer that is divisible by 2 without a remainder.\n"
    "Option A: 3 is not divisible by 2.\n"
    "Option B: 7 is not divisible by 2.\n"
    "Option C: 4 is divisible by 2 ($4 \\div 2 = 2$).\n"
    "Option D: 9 is not divisible by 2.\n"
    "Therefore, the even number is 4, which corresponds to option C.\n"
    "\\\\boxed{C}"
)

def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question

Loaded 1126 questions.


## 3. Model and Sampling Configuration

Here we initialize the model and tokenizer. The key change is in `SamplingParams`:
1.  `n=5`: We instruct vLLM to generate 5 different output sequences for each prompt.
2.  `temperature=0.7`: We use a non-zero temperature to encourage diversity in the generations, which is essential for majority voting to be effective.

In [4]:
import math
from tqdm.notebook import tqdm # Use tqdm.notebook for Jupyter environments

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

print("--- Calculating optimal vLLM parameters ---")

# --- Step 1: Count the maximum input tokens observed across your dataset ---
max_input_tokens_seen = 0
print(f"Analyzing {len(data)} items to determine max input token length...")

# Iterate through your entire dataset to find the prompt with the most tokens.
# We're using the same prompt building logic as your inference loop.
for item in tqdm(data, desc="Tokenizing prompts"):
    system, user = build_prompt(item["question"], item.get("options"))

    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )

    current_input_tokens = len(tokenizer.encode(prompt_text))

    if current_input_tokens > max_input_tokens_seen:
        max_input_tokens_seen = current_input_tokens

print(f"\nMaximum input tokens observed (including chat template): {max_input_tokens_seen}")

# --- Step 2: Define your desired maximum output tokens (from sampling_params) ---
desired_max_output_tokens = 4096 
print(f"Desired maximum output tokens per generation: {desired_max_output_tokens}")

# --- Step 3: Calculate max_model_len for the LLM instance ---
BUFFER_TOKENS = 128 
calculated_max_model_len = max_input_tokens_seen + desired_max_output_tokens + BUFFER_TOKENS
max_model_len_final = max(512, calculated_max_model_len)
print(f"Calculated max_model_len for LLM: {max_model_len_final} (from {max_input_tokens_seen} input + {desired_max_output_tokens} output + {BUFFER_TOKENS} buffer)")

# --- Step 4: Determine max_num_seqs ---
num_generations_per_prompt = 5 
max_num_seqs_final = num_generations_per_prompt
print(f"Setting max_num_seqs to: {max_num_seqs_final} (matches sampling_params.n)")

# --- Step 5: Set max_num_batched_tokens ---
initial_batch_capacity_estimate = max_input_tokens_seen * max_num_seqs_final
max_num_batched_tokens_final = max(32768, initial_batch_capacity_estimate * 2) 
print(f"Initial estimate for max_num_batched_tokens: {max_num_batched_tokens_final}. "
      "This value is heuristic and might require empirical tuning.")


# --- Step 6: Configure the LLM instance with the calculated parameters ---
print("\n--- Configuring LLM with optimized parameters ---")
llm = LLM(
    model=MODEL_ID,
    dtype="bfloat16",
    enable_prefix_caching=False,
    gpu_memory_utilization=0.90,
    max_model_len=max_model_len_final,
    trust_remote_code=True,
    max_num_seqs=max_num_seqs_final,
    max_num_batched_tokens=max_num_batched_tokens_final,
)

# --- Step 7: Configure SamplingParams ---
sampling_params = SamplingParams(
    n=num_generations_per_prompt,
    temperature=0.6,
    top_p=0.95,
    max_tokens=desired_max_output_tokens,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

--- Calculating optimal vLLM parameters ---
Analyzing 1126 items to determine max input token length...


Tokenizing prompts:   0%|          | 0/1126 [00:00<?, ?it/s]


Maximum input tokens observed (including chat template): 4136
Desired maximum output tokens per generation: 4096
Calculated max_model_len for LLM: 8360 (from 4136 input + 4096 output + 128 buffer)
Setting max_num_seqs to: 5 (matches sampling_params.n)
Initial estimate for max_num_batched_tokens: 41360. This value is heuristic and might require empirical tuning.

--- Configuring LLM with optimized parameters ---
INFO 05-27 03:28:18 [utils.py:233] non-default args: {'trust_remote_code': True, 'dtype': 'bfloat16', 'max_model_len': 8360, 'enable_prefix_caching': False, 'max_num_batched_tokens': 41360, 'max_num_seqs': 5, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-4B-Thinking-2507'}


INFO 05-27 03:28:19 [model.py:549] Resolved architecture: Qwen3ForCausalLM


INFO 05-27 03:28:19 [model.py:1678] Using max model len 8360


...

INFO 05-27 03:28:39 [gpu_model_runner.py:6046] Graph capturing finished in 1 secs, took 0.04 GiB


(EngineCore pid=5405) 

INFO 05-27 03:28:39 [gpu_worker.py:597] CUDA graph pool memory: 0.04 GiB (actual), 0.06 GiB (estimated), difference: 0.01 GiB (31.8%).


(EngineCore pid=5405) 

INFO 05-27 03:28:39 [core.py:283] init engine (profile, create kv cache, warmup model) took 11.94 seconds


## 4. Aggregation Logic

This function takes a list of generated responses, extracts the `\boxed{}` answer from each, and returns the most common answer. This is the majority voting mechanism.

In [5]:
def majority_vote_boxed_answer(responses: List[str]) -> str:
    """
    Extracts the last \boxed{} answer from a list of LLM responses and 
    returns the most common answer (majority vote).

    Args:
        responses: A list of string outputs from the language model.

    Returns:
        The most frequently occurring answer. If no boxed answers are found,
        it returns the full text of the first response as a fallback.
    """
    boxed_answers = []
    pattern = re.compile(r"\\boxed{(.*?)}", re.DOTALL)
    
    for response in responses:
        matches = pattern.findall(response)
        if matches:
            # Take the last boxed answer as the final one for this generation
            last_answer = matches[-1].strip()
            boxed_answers.append(last_answer)

    # If no boxed answers were found in any generation, return the first full response
    if not boxed_answers:
        return responses[0] if responses else ""

    # Use Counter to find the most common answer
    vote_counts = Counter(boxed_answers)
    # most_common(1) returns a list like [('answer', count)]
    most_common_answer = vote_counts.most_common(1)[0][0]
    
    return most_common_answer

## 5. Generation and Submission

We now loop through the entire dataset. For each question, we generate 5 responses, aggregate them using our majority vote function, and store the final result. Finally, we save the results to a `.jsonl` file in the required submission format.

In [6]:
submission_records = []

limited_data = data[:10]

print(f"Generating responses for {len(limited_data)} questions...")

# Use tqdm for a progress bar, iterating over the 'limited_data'
for item in tqdm(limited_data, desc="Processing 10 questions"):
# --- CHANGE END ---
    # 1. Build the prompt
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )

    # 2. Generate multiple outputs for the single prompt
    # llm.generate expects a list of prompts
    vllm_outputs = llm.generate([prompt_text], sampling_params, use_tqdm=False)

    # 3. Extract the text from each of the 'n' generations
    # vllm_outputs is a list containing one RequestOutput object
    generated_responses = [output.text.strip() for output in vllm_outputs[0].outputs]

    # 4. Aggregate the answers using majority voting
    final_answer = majority_vote_boxed_answer(generated_responses)

    # 5. Store the result for submission
    submission_records.append({
        "id": item.get("id"),
        "response": final_answer
    })

print("\nGeneration complete.")

Generating responses for 10 questions...


Processing 10 questions:   0%|          | 0/10 [00:00<?, ?it/s]


Generation complete.


### Get the score

In [7]:
import re
import sys
from tqdm.notebook import tqdm # Use tqdm.notebook for Jupyter environments

# --- Scoring Helper Functions ---
# These functions are directly from your baseline notebook.
def extract_letter(text: str) -> str:
    """
    Extracts a single uppercase letter from the response text.
    Prioritizes LaTeX \boxed{} format, then last single uppercase letter.
    """
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""

def score_mcq(response: str, gold_letter: str) -> bool:
    """
    Scores a multiple-choice question by comparing the extracted letter
    from the response to the gold letter.
    """
    return extract_letter(response) == gold_letter.strip().upper()

# --- Judger Setup ---
# Load Judger for free-form scoring.
# Ensure 'judger.py' is in the same directory as your notebook,
# or adjust sys.path.insert accordingly.
# If 'judger.py' is not present, you'll need to obtain it or
# implement a simpler free-form scoring logic.
try:
    sys.path.insert(0, ".") # Temporarily add current directory to Python path
    from judger import Judger
    judger = Judger(strict_extract=False)
    print("Judger loaded successfully for free-form scoring.")
except ImportError:
    print("WARNING: Could not import 'judger.py'. Free-form questions will be marked incorrect.")
    class DummyJudger:
        def auto_judge(self, pred, gold, options):
            return False
    judger = DummyJudger()


# --- Prepare data for efficient lookup ---
# Create a dictionary for quick access to original question details by ID.
data_map = {item["id"]: item for item in data}

# --- Scoring Loop ---
results = []
print(f"Scoring {len(submission_records)} generated responses...")

# Iterate through the generated responses (submission_records)
for sub_record in tqdm(submission_records, desc="Scoring responses"):
    item_id = sub_record["id"]
    response_text = sub_record["response"]
    
    # Retrieve the original question item using its ID
    original_item = data_map.get(item_id)
    
    if original_item is None:
        print(f"WARNING: Original item with ID {item_id} not found in 'data'. Skipping scoring.")
        continue # Skip if original question data is missing
    
    is_mcq = bool(original_item.get("options")) # Check if it's an MCQ
    gold_answer = original_item["answer"] # Get the gold standard answer

    correct = False # Default to incorrect
    if is_mcq:
        correct = score_mcq(response_text, str(gold_answer))
    else:
        # For free-form, gold can be a single answer or a list of acceptable answers
        gold_list = gold_answer if isinstance(gold_answer, list) else [gold_answer]
        try:
            correct = judger.auto_judge(
                pred=response_text,
                gold=gold_list,
                # For free-form, options might not be relevant to Judger, or passed as empty list
                options=[[]] * len(gold_list), # Provide empty options as Judger might expect it
            )
        except Exception as e:
            print(f"Error judging free-form question ID {item_id}: {e}")
            correct = False # Mark as incorrect if judging fails

    results.append({
        "id":       item_id,
        "is_mcq":   is_mcq,
        "gold":     gold_answer,
        "response": response_text,
        "correct":  correct,
    })

print(f"\nScoring complete. {len(results)} results processed.")

# --- Evaluation Summary ---
# These functions and print statements are directly from your baseline notebook.
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    """Calculates accuracy for a subset of results."""
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("\n" + "=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)


Judger loaded successfully for free-form scoring.
Scoring 10 generated responses...


Scoring responses:   0%|          | 0/10 [00:00<?, ?it/s]


Scoring complete. 10 results processed.

EVALUATION RESULTS
  MCQ        :    2 /    3  (66.67%)
  Free-form  :    1 /    7  (14.29%)
  Overall    :    3 /   10  (30.00%)


## 6. Save Results

Save the aggregated responses to the output file.

In [ ]:
from pathlib import Path

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for record in submission_records:
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(submission_records)} records to {out_path}")